# 02 SL/TP Stability Map

Run a small grid over SL/TP parameters. The goal is not to find one beautiful point. The goal is to identify parameter neighborhoods that stay stable.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
project_root = cwd
while not (project_root / "pyproject.toml").exists() and project_root.parent != project_root:
    project_root = project_root.parent
if not (project_root / "pyproject.toml").exists():
    raise RuntimeError("Could not find project root containing pyproject.toml")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

BACKTEST_ROOT = project_root / "backtest_optimize"
RAW_SIGNALS = project_root / "raw_signals"
OUTPUT_DIR = BACKTEST_ROOT / "outputs" / "stability_maps"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import pandas as pd

from backtest_optimize.contracts import AmbiguityPolicy, MarketSpec
from backtest_optimize.io.signal_loader import load_signal_csv
from backtest_optimize.io.market_data import load_ohlcv_from_core
from backtest_optimize.execution.engine import run_single
from backtest_optimize.analysis.metrics import summarize
from backtest_optimize.analysis.optimize import add_stability_scores, run_grid
from backtest_optimize.analysis.versioning import save_snapshot

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

In [ ]:
SYMBOL = "US30"
TIMEFRAME = "H4"
SIGNAL_FILE = RAW_SIGNALS / "combo" / "combo_US30_H4_20230102_20260511_signals.csv"
WARMUP_BARS = 0  # Increase when using swing_extreme SL or other lookback-based methods.

MARKET_SPEC = MarketSpec(symbol=SYMBOL, pip_size=1.0, pip_value_per_lot=1.0, min_lot=0.01, lot_step=0.01)

BASE_CONFIG = {
    "account_size": 10_000.0,
    "risk_per_cluster": 0.01,
    "sl_method": "atr_multiple",
    "tp_method": "risk_multiple",
    "ambiguity_policy": AmbiguityPolicy.CONSERVATIVE,
    "management": {"sl_move_rule": "breakeven_after_tp1"},
}

PARAM_GRID = {
    "atr_mult": [1.0, 1.25, 1.5, 1.75, 2.0],
    "tp1_r": [0.8, 1.0, 1.2],
    "tp2_r": [1.5, 2.0, 2.5],
    "tp3_r": [2.5, 3.0, 3.5],
}

In [ ]:
signals = load_signal_csv(SIGNAL_FILE, symbol=SYMBOL, timeframe=TIMEFRAME)
start = signals["bartime"].min()
end = signals["bartime"].max() + pd.Timedelta(days=10)
bars = load_ohlcv_from_core(SYMBOL, TIMEFRAME, start=start, end=end, warmup_bars=WARMUP_BARS, tail_bars=5)

print(len(signals), len(bars))

In [ ]:
def evaluate(params):
    if params["tp1_r"] >= params["tp2_r"] or params["tp2_r"] >= params["tp3_r"]:
        return {"expectancy_r": None, "skip_rate": None, "invalid_params": True}

    result = run_single(
        signals=signals,
        bars=bars,
        symbol=SYMBOL,
        timeframe=TIMEFRAME,
        market_spec=MARKET_SPEC,
        sl_params={"atr_mult": params["atr_mult"]},
        tp_params={"r_multiples": [params["tp1_r"], params["tp2_r"], params["tp3_r"]]},
        **BASE_CONFIG,
    )
    summary = summarize(result)
    summary["invalid_params"] = False
    return summary

grid = run_grid(evaluate, PARAM_GRID)
stable = add_stability_scores(
    grid,
    metric_col="expectancy_r",
    param_cols=["atr_mult", "tp1_r", "tp2_r", "tp3_r"],
    neighborhood_pct=0.15,
)

display(stable.sort_values("expectancy_r_stability_score", ascending=False).head(20))

In [ ]:
run_name = f"stability_{SYMBOL}_{TIMEFRAME}_{pd.Timestamp.now('UTC').strftime('%Y%m%d_%H%M%S')}"
path = OUTPUT_DIR / f"{run_name}.csv"
stable.to_csv(path, index=False)

snapshot_path = save_snapshot(
    name=run_name,
    config={
        "symbol": SYMBOL,
        "timeframe": TIMEFRAME,
        "base_config": BASE_CONFIG,
        "market_spec": MARKET_SPEC,
        "param_grid": PARAM_GRID,
        "warmup_bars": WARMUP_BARS,
    },
    result_summary={
        "grid_rows": len(stable),
        "valid_rows": int((stable["invalid_params"] == False).sum()),
        "invalid_rows": int((stable["invalid_params"] == True).sum()),
        "best_stability_score": stable["expectancy_r_stability_score"].max(),
        "output_csv": str(path),
    },
    signal_file=SIGNAL_FILE,
    market_data_source_id="core_python.data.loader",
    repo_root=project_root,
)

print(path)
print(snapshot_path)